# Part 4: Add an authenticated MCP source (GitHub)

In Part 3, you connected a non-authenticated MCP source (Microsoft Learn). In this part, you'll add an **authenticated** MCP source — [GitHub's MCP server](https://github.com/github/github-mcp-server) — which requires a Bearer token.

The key difference: authentication headers are provided **at query time** via `knowledgeSourceParams.headers`, not when creating the source. This means the token stays with the caller, not stored in the service.

## Step 0: Configure GitHub authentication

Before running the code cells in this notebook, add `GITHUB_TOKEN` to your `.env` file. Use either a GitHub personal access token (classic) with `repo` scope, or a fine-grained token with appropriate repository permissions. For setup guidance, see [Managing your personal access tokens](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

## Step 1: Load environment variables

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
api_key = os.environ["AZURE_SEARCH_ADMIN_KEY"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
github_token = os.environ["GITHUB_TOKEN"]
azure_openai_chatgpt_deployment = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
azure_openai_chatgpt_model_name = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

knowledge_base_name = "hr-and-health-docs-knowledge-base"
api_version = "2025-11-01-preview"

headers = {"Content-Type": "application/json", "api-key": api_key}

print("Environment variables loaded")

## Step 2: Create GitHub MCP knowledge source

Create a knowledge source pointing to GitHub's MCP server at `https://api.githubcopilot.com/mcp`. The `toolName` specifies `search_issues` — the MCP tool for searching across GitHub issues and pull requests.

Note that no auth headers are set here — they're provided at query time.

In [ ]:
import requests

github_ks_name = "github-issues-source"

github_ks_body = {
    "name": github_ks_name,
    "kind": "mcpTool",
    "description": "GitHub MCP for searching issues and pull requests across repositories",
    "mcpToolParameters": {
        "serverURL": "https://api.githubcopilot.com/mcp",
        "toolName": "search_issues"
    }
}

url = f"{endpoint}/knowledgesources/{github_ks_name}?api-version={api_version}"
response = requests.put(url, json=github_ks_body, headers=headers)
response.raise_for_status()
print(f"Knowledge source '{github_ks_name}' created: {response.status_code}")

## Step 3: Add GitHub source to the knowledge base

Update the knowledge base to include all four sources: hrdocs, healthdocs, Learn MCP, and GitHub MCP.

In [ ]:
kb_body = {
    "name": knowledge_base_name,
    "description": "Knowledge base combining enterprise docs, Microsoft Learn, and GitHub issues",
    "retrievalInstructions": (
        "Use healthdocs for health benefits and insurance questions. "
        "Use hrdocs for HR policies and company information. "
        "Use learn-mcp-source for Azure, Microsoft product, or technical documentation questions. "
        "Use github-issues-source for GitHub issues, bugs, feature requests, or pull request discussions."
    ),
    "outputMode": "answerSynthesis",
    "knowledgeSources": [
        {"name": "healthdocs-knowledge-source"},
        {"name": "hrdocs-knowledge-source"},
        {"name": "learn-mcp-source"},
        {"name": github_ks_name}
    ],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": azure_openai_endpoint,
                "deploymentId": azure_openai_chatgpt_deployment,
                "modelName": azure_openai_chatgpt_model_name,
                "apiKey": azure_openai_key
            }
        }
    ]
}

url = f"{endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}"
response = requests.put(url, json=kb_body, headers=headers)
response.raise_for_status()
print(f"Knowledge base '{knowledge_base_name}' updated: {response.status_code}")

## Step 4: Query with authenticated GitHub source

When querying, pass the GitHub Bearer token in `knowledgeSourceParams.headers` for the GitHub source. This authenticates the request at query time without storing credentials in the service.

Ask about a specific GitHub repository's issues to see the authenticated source in action:

In [ ]:
from IPython.display import Markdown, display

retrieve_url = f"{endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}"

retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What are the recent open issues in the PrefectHQ/fastmcp repository?"
                }
            ]
        }
    ],
    "knowledgeSourceParams": [
        {
            "kind": "mcpTool",
            "knowledgeSourceName": github_ks_name,
            "headers": {
                "Authorization": f"Bearer {github_token}"
            }
        }
    ],
    "includeActivity": True
}

response = requests.post(retrieve_url, json=retrieve_body, headers=headers)
response.raise_for_status()
result = response.json()
display(Markdown(result["response"][0]["content"][0]["text"]))

## Step 5: Verify source routing

Check the activity log to confirm the GitHub MCP source was queried:

In [ ]:
import json

print("Sources queried:")
for activity in result.get("activity", []):
    ks_name = activity.get("knowledgeSourceName", "")
    activity_type = activity.get("type", "")
    if ks_name:
        print(f"  [{activity_type}] → {ks_name}")

print("\nFull activity log:")
print(json.dumps(result.get("activity", []), indent=2))

## Step 6: Combined query — all sources

Ask a question that touches enterprise data, Microsoft documentation, and GitHub issues. The agent routes subqueries to the appropriate sources:

In [ ]:
retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "What does the Zava CEO do? "
                        "What are the recent open issues in the PrefectHQ/fastmcp repository? "
                        "How do I set up agentic retrieval in Azure AI Search?"
                    )
                }
            ]
        }
    ],
    "knowledgeSourceParams": [
        {
            "kind": "mcpTool",
            "knowledgeSourceName": github_ks_name,
            "headers": {
                "Authorization": f"Bearer {github_token}"
            }
        }
    ],
    "includeActivity": True
}

response = requests.post(retrieve_url, json=retrieve_body, headers=headers)
response.raise_for_status()
result = response.json()
display(Markdown(result["response"][0]["content"][0]["text"]))

print("\nSources queried:")
for activity in result.get("activity", []):
    ks_name = activity.get("knowledgeSourceName", "")
    if ks_name:
        print(f"  → {ks_name}")

## Summary

You've added an authenticated MCP knowledge source to your knowledge base. The GitHub MCP server requires a Bearer token at query time, demonstrating the pattern for any MCP server that needs authentication.

**Key concepts:**
- Authenticated MCP sources use the same `kind: mcpTool` definition as unauthenticated ones
- Auth headers are passed at **query time** via `knowledgeSourceParams.headers`, not at creation time
- This keeps credentials with the caller, not stored in the service
- The knowledge base can now route across indexed data, Learn docs, and GitHub issues simultaneously

➡️ Continue to [Part 5: Connect KB to a Foundry agent](part5-foundry-agent.ipynb) to wire this knowledge base into a Microsoft Foundry Agent for conversational grounding.